# 08 — Qualitätsschranke und Modellregister

**Ziel dieses Notebooks** ist, dass ein Modell nur dann gespeichert wird, wenn es
Mindestanforderungen erfüllt.

Bisher wird überschreibt jeder Trainingslauf `models/model.joblib` überschrieben. 

Die Schranke macht aus einer besprochenen Anforderung eine **ausgeführte**.

In [1]:
from pathlib import Path

import mlflow
import pandas as pd
from mlflow import MlflowClient

from src.data.validate import load_params
from src.modeling.quality_gate import pruefe_qualitaet
from src.modeling.train import EXPERIMENT, TRACKING_URI, trainiere

mlflow.set_tracking_uri(TRACKING_URI)
params = load_params()
schranken = params["quality_gate"]
registername = params["model"]["registry_name"]

print("Anforderungen aus params.yaml:")
for name, wert in schranken.items():
    print(f"  {name:24s} {wert}")

/Users/marzenakanarek/Documents/GitHub/ML Operationalisierung/ST_MLops/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Anforderungen aus params.yaml:
  min_pr_auc               0.75
  min_recall               0.8
  max_expected_cost_eur    200000


## 1 Prüfund

Drei Anforderungen:

**`min_pr_auc: 0.75`** — die Erfolgsschwelle aus der README, vor dem ersten
Training festgelegt. Sie misst die Modellgüte unabhängig vom Schwellenwert.

**`min_recall: 0.80`** — wie viele der tatsächlichen Ausfälle gefunden werden.
Ein Modell mit guter PR-AUC, das am gewählten Schwellenwert zu zurückhaltend
ist, soll trotzdem durchfallen.

**`max_expected_cost_eur: 200000`** — die Übersetzung in Euro. Sie fängt
Fälle ab, in denen die beiden anderen Zahlen knapp passen, das Verhältnis von
übersehenen Ausfällen zu Fehlalarmen aber aus dem Ruder läuft.

Geprüft wird auf der **Validierungsmenge**, beim festgelegten
Entscheidungsschwellenwert von 0,06. Die Testmenge bleibt weiterhin unangetastet.

In [2]:
# Die Prüffunktion arbeitet auf einem Kennzahlen-Wörterbuch und ist damit
# unabhängig vom Training testbar.
beispiel_gut = {"pr_auc": 0.860, "recall": 0.863, "kosten_eur": 126200}
beispiel_schlecht = {"pr_auc": 0.610, "recall": 0.420, "kosten_eur": 310000}

print("── gutes Modell ──")
print(pruefe_qualitaet(beispiel_gut, schranken).bericht())
print()
print("── schlechtes Modell ──")
print(pruefe_qualitaet(beispiel_schlecht, schranken).bericht())

── gutes Modell ──
3 Anforderungen geprüft.
  erfüllt: PR-AUC
  erfüllt: Recall
  erfüllt: erwartete Kosten

Alle Anforderungen erfüllt — das Modell darf gespeichert werden.

── schlechtes Modell ──
3 Anforderungen geprüft.

NICHT ERFÜLLT (3):
  - PR-AUC — 0.610 liegt unter der Mindestanforderung 0.75
  - Recall — 0.420 liegt unter der Mindestanforderung 0.8
  - erwartete Kosten — 310.000 EUR übersteigen die Obergrenze 200.000 EUR


Alle Verstöße werden gesammelt.

In [3]:
modell_pfad = Path(params["paths"]["models_dir"]) / "model.joblib"

ergebnis = trainiere(lauf_name="gate_bestanden")
print("Schranke bestanden:", ergebnis["schranke_bestanden"])
print(f"PR-AUC {ergebnis['pr_auc']:.3f} | Recall {ergebnis['recall']:.3f} | "
      f"Kosten {ergebnis['kosten_eur']:,.0f} EUR".replace(",", "."))

zeitstempel_vorher = modell_pfad.stat().st_mtime
print(f"\nmodels/model.joblib geschrieben, Größe "
      f"{modell_pfad.stat().st_size / 1024:.0f} KB")

2026/10/02 10:50:09 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Schranke bestanden: True
PR-AUC 0.860 | Recall 0.863 | Kosten 126.200 EUR

models/model.joblib geschrieben, Größe 886 KB


Registered model 'predictive-maintenance' already exists. Creating a new version of this model...
Created version '9' of model 'predictive-maintenance'.


## 3 Gegenprobe

Ein absichtlich verschlechtertes Modell: `max_depth=1` wird abgelehnt. Das bisherige Modell bleibt **unverändert**.

In [4]:
schlecht = trainiere(lauf_name="gate_durchgefallen",
                     ueberschreibungen={"max_depth": 1})

print("Schranke bestanden:", schlecht["schranke_bestanden"])
print(f"PR-AUC {schlecht['pr_auc']:.3f} | Recall {schlecht['recall']:.3f} | "
      f"Kosten {schlecht['kosten_eur']:,.0f} EUR".replace(",", "."))
print()
for verstoss in schlecht["verstoesse"]:
    print(f"  - {verstoss}")

zeitstempel_nachher = modell_pfad.stat().st_mtime
print()
print("models/model.joblib unverändert:", zeitstempel_vorher == zeitstempel_nachher)
assert zeitstempel_vorher == zeitstempel_nachher, \
    "Das durchgefallene Modell hat die Datei überschrieben!"

2026/10/02 10:50:11 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Schranke bestanden: False
PR-AUC 0.647 | Recall 0.451 | Kosten 303.400 EUR

  - PR-AUC — 0.647 liegt unter der Mindestanforderung 0.75
  - Recall — 0.451 liegt unter der Mindestanforderung 0.8
  - erwartete Kosten — 303.400 EUR übersteigen die Obergrenze 200.000 EUR

models/model.joblib unverändert: True


Ein absichtlich verschlechtertes Modell wird abgelehnt und nicht gespeichert.

Die `assert`-Zeile ist bewusst gesetzt. Sollte jemand den Ablauf später so
umbauen, dass zuerst gespeichert und dann geprüft wird, bricht dieses Notebook
ab — statt dass der Fehler erst im Betrieb auffällt.

Auf der Kommandozeile endet ein durchgefallener Lauf zusätzlich mit dem
Fehlercode 1. Das ist für die CI entscheidend: Ein Arbeitsablauf,
der `python -m src.modeling.train` aufruft, schlägt dann automatisch fehl.

## 4 Modellregister

Registriert wird nur, was die Schranke bestanden hat. Das bestandene Modell
bekommt den Alias **`champion`**.

In [5]:
klient = MlflowClient()

versionen = klient.search_model_versions(f"name='{registername}'")
tabelle = pd.DataFrame([
    {"Version": int(v.version), "Lauf": v.run_id[:8],
     "Status": v.status, "erstellt": pd.to_datetime(v.creation_timestamp, unit="ms")}
    for v in versionen
]).sort_values("Version")

champion = klient.get_model_version_by_alias(registername, "champion")
print(f"Registername: {registername}")
print(f"champion zeigt auf Version {champion.version}")
print(f"Anzahl registrierter Versionen: {len(tabelle)}")
tabelle

Registername: predictive-maintenance
champion zeigt auf Version 9
Anzahl registrierter Versionen: 9


,Version,Lauf,Status,erstellt
8,1,c41d2317,READY,2026-10-02 07:58:39.559
7,2,ea7e14ff,READY,2026-10-02 07:58:41.285
6,3,527f47db,READY,2026-10-02 07:59:45.641
5,4,2e9610b6,READY,2026-10-02 08:35:24.020
4,5,f5a3ceee,READY,2026-10-02 08:35:25.858
3,6,d167def4,READY,2026-10-02 08:36:07.359
2,7,533df27a,READY,2026-10-02 08:49:55.701
1,8,489f9bed,READY,2026-10-02 08:49:57.358
0,9,65a632b9,READY,2026-10-02 08:50:11.362


## 5 Alias statt Dateipfad

Die Schnittstelle könnte `models/model.joblib` laden. Dann hieße
„das aktuelle Modell" aber *die Datei, die zuletzt jemand dorthin geschrieben
hat* — ohne Historie und ohne Weg zurück.

Mit dem Register lädt sie stattdessen:

```python
mlflow.sklearn.load_model(f"models:/{registername}@champion")
```

Das bringt drei Dinge mit:

**Rückfall ist ein Handgriff.** Erweist sich eine neue Version im Betrieb als
schlechter, zeigt man den Alias wieder auf die alte — kein Dateikopieren, kein
Neustart mit vertauschten Dateien.

**Die Historie bleibt.** Jede Version ist mit ihrem Lauf verknüpft, und über den
Lauf mit Datenhash, Git-Commit und Kennzahlen. 

**Der Name ist die Schnittstelle.** Der Dienst kennt nur `@champion`. Welche
Version das gerade ist, entscheidet das Register.

In [6]:
# Rückfall vorführen: Alias auf eine frühere Version setzen und zurück.
alle_versionen = sorted(int(v.version) for v in versionen)
print("vorhandene Versionen:", alle_versionen)

if len(alle_versionen) >= 2:
    vorherige, aktuelle = alle_versionen[-2], alle_versionen[-1]
    klient.set_registered_model_alias(registername, "champion", str(vorherige))
    print(f"champion -> Version "
          f"{klient.get_model_version_by_alias(registername, 'champion').version}  (Rückfall)")
    klient.set_registered_model_alias(registername, "champion", str(aktuelle))
    print(f"champion -> Version "
          f"{klient.get_model_version_by_alias(registername, 'champion').version}  (zurück)")
else:
    print("Für die Vorführung des Rückfalls braucht es mindestens zwei Versionen.")
    print("Nach einem weiteren bestandenen Lauf steht sie zur Verfügung.")

vorhandene Versionen: [1, 2, 3, 4, 5, 6, 7, 8, 9]
champion -> Version 8  (Rückfall)
champion -> Version 9  (zurück)


---

## Summary

| | |
|---|---|
| Geprüft | PR-AUC ≥ 0,75 · Recall ≥ 0,80 · Kosten ≤ 200.000 € |
| Geprüft auf | Validierungsmenge, Schwellenwert 0,06 |
| Bei Nichtbestehen | kein Speichern, keine Registrierung, Fehlercode 1 |
| Bei Bestehen | `models/model.joblib` plus Registrierung mit Alias `champion` |
| Gegenprobe | `max_depth=1` fällt an allen drei Anforderungen durch |

Die Mindestanforderungen stehen in `params.yaml` und werden ausgeführt. Ein durchgefallenes Modell kann das gute nicht überschreiben. Das ausgelieferte Modell wird über einen Alias angesprochen statt über einen Dateipfad, damit auf die Vorgängerversion zugreifen kann.